# Does terrain remove the remaining false alarms?

[Notebook 04](04_false_alarms_s1.ipynb) halved the false alarms with the pre-event pass and a minimum patch size. Most of what is left got darker between the two passes, far from the mapped flood. Water stays low and flat, so terrain looks like the obvious next filter: drop what lies on a slope or high above the valley floor.

This notebook tests that with the [Copernicus DEM GLO-30](https://registry.opendata.aws/copernicus-dem/), a public dataset that spends **0 PU**. It reads about 400 windows over the network the first time (about 11 minutes), then reuses them from the data directory. Everything else comes from notebook 02's pairs.

Two terrain measures, on the 40 m grid of the chips:

* **slope**, in degrees;
* **height above the local low**: elevation minus the lowest elevation within about 1 km. It is a cheap stand-in for HAND, not HAND itself: it does not follow the drainage network.

In [ ]:
import csv
import math
import os
from pathlib import Path
from urllib.request import urlopen

import numpy as np

ACTIVATIONS = [
    "EMSR659",
    "EMSR664",
    "EMSR705",
    "EMSR762",
    "EMSR795",
    "EMSR800",
    "EMSR843",
    "EMSR855",
]
RESOLUTION_M = 40
CHIP = 128
SEED = 0
MARGIN = 32  # pixels of terrain read around a chip, for the local low
LOW_WINDOW = 51  # pixels, about 2 km across

DATA_DIR = Path(os.environ.get("COPERNICUS_EMS_DATA_DIR", "~/.cache/copernicus-ems")).expanduser()
DATASET = DATA_DIR / "datasets" / f"italy-floods-s1-change-{RESOLUTION_M}m"
DEM_DIR = DATA_DIR / f"dem-{RESOLUTION_M}m"
FIGURE = Path(os.environ.get("NOTEBOOK_FIGURE", "../docs/images/notebook-terrain-s1.png"))
print("data:", DATA_DIR, "| dataset:", DATASET)

## 1. The five-band dataset of notebook 02

The dataset is rebuilt from the pairs in the data directory, locally and for free. Its train/val/test column is ignored: the folds are made by activation below.

In [ ]:
from copernicus_ems.dataset import build_dataset, find_pairs

pairs = find_pairs(DATA_DIR, ACTIVATIONS)
assert any(p.reference is not None for p in pairs), (
    "no pair with a reference: run notebook 02 first"
)
summary = build_dataset(pairs, DATASET, size=CHIP, min_observed=0.5, seed=SEED, reference=True)
rows = list(csv.DictReader((DATASET / "index.csv").open()))
made_from = {
    (p.activation, p.product): sorted({i["sensor"] for i in p.meta.get("made_from", [])})
    for p in pairs
}
for r in rows:
    r["made_from"] = ", ".join(made_from.get((r["activation"], r["product"]), []))
    r["same_image"] = "Sentinel-1" in r["made_from"]
FOLDS = sorted({r["activation"] for r in rows})
print(
    f"{summary.pairs} pairs -> {summary.chips} chips of {CHIP} px; {len(FOLDS)} folds: {', '.join(FOLDS)}"
)

## 2. Features

The eleven features of notebook 02: VV and VH in dB, their ratio, local mean and spread of VV, the pre-event VV and VH, and the changes ΔVV, ΔVH, raw and smoothed over 5×5 pixels.

In [ ]:
import rasterio
from scipy.ndimage import uniform_filter

from copernicus_ems.ml import FLOODED, IGNORE, LabelRaster

SINGLE = slice(0, 5)
NAMES = [
    "VV",
    "VH",
    "VV-VH",
    "VV mean",
    "VV spread",
    "VV pre",
    "VH pre",
    "dVV",
    "dVH",
    "dVV mean",
    "dVH mean",
]


def db(x):
    return 10 * np.log10(np.maximum(x, 1e-6))


def features(path):
    with rasterio.open(path) as src:
        vv, vh, vv_pre, vh_pre, _mask = src.read()
    vv_db, vh_db, vvp_db, vhp_db = db(vv), db(vh), db(vv_pre), db(vh_pre)
    mean = uniform_filter(vv_db, 5)
    spread = np.sqrt(np.maximum(uniform_filter(vv_db**2, 5) - mean**2, 0))
    dvv, dvh = vv_db - vvp_db, vh_db - vhp_db
    return np.stack(
        [
            vv_db,
            vh_db,
            vv_db - vh_db,
            mean,
            spread,
            vvp_db,
            vhp_db,
            dvv,
            dvh,
            uniform_filter(dvv, 5),
            uniform_filter(dvh, 5),
        ],
        axis=-1,
    ).astype(np.float32)


X = np.stack([features(DATASET / r["image"]) for r in rows])
LABELS = np.stack([LabelRaster.from_geotiff(DATASET / r["label"]).labels for r in rows])
Y = [LabelRaster.from_geotiff(DATASET / r["label"]) for r in rows]
km2 = (RESOLUTION_M / 1000) ** 2
print(
    f"{'activation':10} {'chips':>5} {'flooded km²':>12} {'independent km²':>16} {'same image km²':>15}"
)
for a in FOLDS:
    idx = [i for i, r in enumerate(rows) if r["activation"] == a]
    wet = [int((LABELS[i] == FLOODED).sum()) for i in idx]
    ind = sum(w for i, w in zip(idx, wet, strict=True) if not rows[i]["same_image"])
    print(
        f"{a:10} {len(idx):5d} {sum(wet) * km2:12.1f} {ind * km2:16.1f} {(sum(wet) - ind) * km2:15.1f}"
    )

## 3. Terrain under every chip

Each chip gets its DEM window, resampled to its grid with a margin. A tile absent from the published list is sea and stays empty; any other read error stops the notebook rather than leaving a silent hole.

In [ ]:
from rasterio.enums import Resampling
from rasterio.transform import from_origin
from rasterio.warp import reproject, transform_bounds
from scipy.ndimage import minimum_filter

BUCKET = "https://copernicus-dem-30m.s3.amazonaws.com"
with urlopen(f"{BUCKET}/tileList.txt") as answer:
    TILES = set(answer.read().decode().split())


def tile_name(lat, lon):
    ns = f"{'N' if lat >= 0 else 'S'}{abs(lat):02d}"
    ew = f"{'E' if lon >= 0 else 'W'}{abs(lon):03d}"
    return f"Copernicus_DSM_COG_10_{ns}_00_{ew}_00_DEM"


def dem_window(row):
    path = DEM_DIR / f"{row['id']}.tif"
    if path.is_file():
        with rasterio.open(path) as src:
            return src.read(1)
    res = float(row["resolution_m"])
    size = int(row["size"]) + 2 * MARGIN
    x0, y0 = float(row["x0"]) - MARGIN * res, float(row["y0"]) + MARGIN * res
    transform = from_origin(x0, y0, res, res)
    west, south, east, north = transform_bounds(
        row["crs"], "EPSG:4326", x0, y0 - size * res, x0 + size * res, y0
    )
    z = np.full((size, size), np.nan, np.float32)
    for lat in range(math.floor(south), math.floor(north) + 1):
        for lon in range(math.floor(west), math.floor(east) + 1):
            name = tile_name(lat, lon)
            if name not in TILES:
                continue
            part = np.full((size, size), np.nan, np.float32)
            with rasterio.open(f"/vsicurl/{BUCKET}/{name}/{name}.tif") as src:
                reproject(
                    rasterio.band(src, 1),
                    part,
                    dst_transform=transform,
                    dst_crs=row["crs"],
                    resampling=Resampling.average,
                    dst_nodata=np.nan,
                )
            z = np.where(np.isnan(z), part, z)
    DEM_DIR.mkdir(parents=True, exist_ok=True)
    with rasterio.open(
        path,
        "w",
        driver="GTiff",
        width=size,
        height=size,
        count=1,
        dtype="float32",
        crs=row["crs"],
        transform=transform,
        nodata=np.nan,
    ) as dst:
        dst.write(z, 1)
    return z


def terrain(row):
    z = dem_window(row)
    z = np.nan_to_num(z, nan=0.0)  # sea level where no tile exists
    grad_y, grad_x = np.gradient(z, float(row["resolution_m"]))
    slope = np.degrees(np.arctan(np.hypot(grad_x, grad_y)))
    low = z - minimum_filter(z, size=LOW_WINDOW)
    crop = (slice(MARGIN, -MARGIN), slice(MARGIN, -MARGIN))
    return slope[crop], low[crop]


with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR"):
    both = [terrain(r) for r in rows]
SLOPE = np.stack([b[0] for b in both])
LOW = np.stack([b[1] for b in both])
print(f"terrain for {len(rows)} chips in {DEM_DIR}")

## 4. Two detectors, tuned on the other activations

* **Not dark before, min size**: the best detector of notebook 04.
* **+ terrain**: the same, also requiring slope < *s*° and height above the local low < *h* m.

In each fold the thresholds, then *s* and *h*, then the patch size are chosen for the best CSI on the training chips only. Terrain may also choose to do nothing: the largest *s* and *h* on the grid keep almost every pixel.

In [ ]:
from scipy.ndimage import label as connected

from copernicus_ems.ml import Scores, evaluate

OBSERVED = LABELS != IGNORE
WET = LABELS == FLOODED
EIGHT = np.ones((3, 3), dtype=bool)
VV, PRE = X[..., 0], X[..., 5]
SIZES = (1, 2, 4, 8, 16, 32)


def csi(chips, predicted):
    hits = int((predicted & WET[chips]).sum())
    false = int((predicted & OBSERVED[chips] & ~WET[chips]).sum())
    return hits / max(int(WET[chips].sum()) + false, 1)


def min_size(predicted, n):
    if n <= 1:
        return predicted
    labelled, _ = connected(predicted, structure=EIGHT)
    big = np.bincount(labelled.ravel()) >= n
    big[0] = False
    return big[labelled]


def sized(mask, n):
    return np.stack([min_size(m, n) for m in mask])


def fold(test_activation):
    train = np.flatnonzero(np.array([r["activation"] != test_activation for r in rows]))
    t, tp = max(
        [(a, b) for a in np.arange(-22, -10, 0.5) for b in np.arange(-22, -8, 1.0)],
        key=lambda c: csi(train, (VV[train] < c[0]) & (PRE[train] >= c[1])),
    )
    base = (VV[train] < t) & (PRE[train] >= tp)
    n_base = max(SIZES, key=lambda k: csi(train, sized(base, k)))
    s, h = max(
        [(a, b) for a in (2, 3, 5, 8, 12, 20, 90) for b in (5, 10, 15, 20, 30, 50, 100, 1e6)],
        key=lambda c: csi(train, base & (SLOPE[train] < c[0]) & (LOW[train] < c[1])),
    )
    flat = base & (SLOPE[train] < s) & (LOW[train] < h)
    n_flat = max(SIZES, key=lambda k: csi(train, sized(flat, k)))
    return {"t": t, "tp": tp, "n_base": n_base, "s": s, "h": h, "n_flat": n_flat}


DETECTORS = ["not dark before, min size", "+ terrain"]
SUBSETS = {"independent": lambda r: not r["same_image"], "same image": lambda r: r["same_image"]}
per_fold, predictions = {}, {}
for a in FOLDS:
    p = fold(a)
    test = [i for i, r in enumerate(rows) if r["activation"] == a]
    for i in test:
        base = (VV[i] < p["t"]) & (PRE[i] >= p["tp"])
        predictions[DETECTORS[0], i] = min_size(base, p["n_base"])
        predictions[DETECTORS[1], i] = min_size(
            base & (SLOPE[i] < p["s"]) & (LOW[i] < p["h"]), p["n_flat"]
        )
    for name in DETECTORS:
        for subset, keep in SUBSETS.items():
            chosen = [i for i in test if keep(rows[i])]
            per_fold[a, name, subset] = sum(
                (evaluate(predictions[name, i], Y[i]) for i in chosen), Scores.zero()
            )
    print(
        f"{a}: VV < {p['t']:.1f}, VV_pre >= {p['tp']:.0f} dB | slope < {p['s']} deg, low < {p['h']:g} m | size {p['n_base']} / {p['n_flat']} px"
    )

## 5. Scores, every flood counted once

In [ ]:
pooled = {
    (name, subset): sum((per_fold[a, name, subset] for a in FOLDS), Scores.zero())
    for name in DETECTORS
    for subset in SUBSETS
}
print(
    f"{'':27} {'labels':12} {'CSI':>6} {'POD':>6} {'FAR':>6} {'F1':>6} {'bias':>6} {'false km²':>10}"
)
for subset in SUBSETS:
    for name in DETECTORS:
        s = pooled[name, subset]
        print(
            f"{name:27} {subset:12} {s.csi:6.3f} {s.pod:6.3f} {s.far:6.3f} {s.f1:6.3f} {s.bias:6.2f} {s.false_alarms * km2:10.1f}"
        )

## 6. Why: false alarms and water share the same ground

Terrain can separate them only if they lie on different ground. Below, the terrain under the mapped flood and under the false alarms of the detector without terrain, on independent labels.

In [ ]:
independent = np.array([SUBSETS["independent"](r) for r in rows])
PREDICTED = np.stack([predictions[DETECTORS[0], i] for i in range(len(rows))])
masks = {
    "mapped flood": WET & independent[:, None, None],
    "false alarms": PREDICTED & OBSERVED & ~WET & independent[:, None, None],
}
print(
    f"{'':14} {'km²':>7} {'slope p50':>10} {'p95':>6} {'p99':>6} {'low p50':>9} {'p95':>6} {'p99':>6}"
)
for name, mask in masks.items():
    a = np.percentile(SLOPE[mask], [50, 95, 99])
    b = np.percentile(LOW[mask], [50, 95, 99])
    print(
        f"{name:14} {mask.sum() * km2:7.1f} {a[0]:10.1f} {a[1]:6.1f} {a[2]:6.1f} {b[0]:9.1f} {b[1]:6.1f} {b[2]:6.1f}"
    )

## 7. Figure

Share of the pixels below each slope and height, for the mapped flood and the false alarms. Where the two curves overlap, no terrain threshold can keep one and drop the other.

In [ ]:
import matplotlib.pyplot as plt

COLORS = {"mapped flood": "#2a78d6", "false alarms": "#eb6834"}
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
panels = [(SLOPE, "slope (degrees)", 10), (LOW, "height above the local low (m)", 30)]
for ax, (values, title, limit) in zip(axes, panels, strict=True):
    for name, mask in masks.items():
        sample = np.sort(values[mask])
        ax.plot(
            sample,
            np.arange(1, len(sample) + 1) / len(sample),
            color=COLORS[name],
            linewidth=2,
            label=name,
        )
    ax.set_xlim(0, limit)
    ax.set_ylim(0, 1)
    ax.set_xlabel(title)
    ax.grid(color="#e5e5e5")
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
axes[0].set_ylabel("share of pixels below")
axes[1].legend(frameon=False, loc="lower right")
fig.text(
    0.5,
    0.005,
    "Contains modified Copernicus Sentinel data, Copernicus EMS information and Copernicus DEM GLO-30, © European Union, ESA, DLR, Airbus",
    ha="center",
    fontsize=7,
)
fig.tight_layout(rect=(0, 0.03, 1, 1))
FIGURE.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURE, dpi=110)
print("saved", FIGURE)

## Reading the scores

* **Terrain does not help here.** Slope and height above the local low leave the score unchanged or slightly worse: the false alarms lie on ground as flat and low as the mapped flood (section 6), so a threshold cannot keep one and drop the other.
* **The measure is a stand-in.** Height above the local low does not follow the drainage network. A true HAND might separate a floodplain from a low field far from any river; this test cannot say.
* **This is an Italian, mostly lowland sample.** Most of the mapped area is in the Romagna plain; in mountain valleys terrain may matter more.

*Data: Copernicus EMS Rapid Mapping, © European Union; Copernicus Sentinel-1, processed by Sentinel Hub on the Copernicus Data Space Ecosystem; Copernicus DEM GLO-30, © DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018, provided under COPERNICUS by the European Union and ESA.*